# SHAP for the saved RoBERTa-base (class weights, seed 42)

Explains the model saved in `Exp_1_4_run_tram_2_epoc_30/roberta_pw_s42`. No training is performed; the notebook reads from Drive and writes to `Exp_3_0_shap`.

**Explained quantity.** The logit (score before the sigmoid) of the model's top-1 predicted label for a test sentence. No probability threshold is used.

**Sentence selection.** The 725 test rows are put in a random order (seed 42) and the first `N_SAMPLES` rows are explained. The rule does not depend on correctness, label or explanation content. Raising `N_SAMPLES` keeps the same first rows.

**Unit of explanation.** Whitespace-delimited words (punctuation stays attached to its word). A hidden word is replaced by the model's `<mask>` token. The whitespace around it is kept, so a sentence with nothing hidden is character-for-character identical to the original (checked by an assertion).

**Algorithm.** SHAP permutation explainer with a fixed seed and `N_PERM` permutation passes per sentence. Values are Monte Carlo estimates; Cell 4 measures how much they change with another seed. The base value is the model score when every word is replaced by `<mask>`.

**Output file.** `shap_words_roberta_pw_s42.jsonl`, one JSON line per sentence, with the method settings stored in every line. Re-running skips rows that are already stored. The earlier file `shap_roberta_pw_s42.jsonl` (SHAP text masker, sub-word units) is not read or modified.

In [ ]:
# Install shap if it is missing (tested with shap 0.54.1)
!pip install -q shap

In [ ]:
# Cell 1 - imports and Drive
import os, re, json, time
import numpy as np
import pandas as pd
import torch
import shap
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


In [ ]:
# Cell 2 - paths, settings, data and model loading, explanation functions
DATA_DIR  = "/content/drive/MyDrive/KDU/Thesis/Dataset"
MODEL_DIR = "/content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30/roberta_pw_s42"
OUT_DIR   = "/content/drive/MyDrive/KDU/Thesis/Exp_3_0_shap"
OUT_FILE  = f"{OUT_DIR}/shap_words_roberta_pw_s42.jsonl"
os.makedirs(OUT_DIR, exist_ok=True)

# Sentence selection rule (fixed before any explanation was inspected):
# random order of the 725 test rows (seed 42); the first N_SAMPLES rows of that order are explained.
SEED, N_SAMPLES = 42, 100
MAX_LENGTH = 128                 # same as training
N_PERM = 10                      # permutation passes per sentence (each pass evaluates a permutation and its reverse)
SHAP_SEED = 42                   # seed of the permutation explainer
PLOT_FIRST = 3                   # number of newly explained rows that are also drawn

test_df = pd.read_csv(f"{DATA_DIR}/tram_test.tsv", sep="\t").rename(columns={"text1": "text"})
texts   = test_df["text"].tolist()
classes = np.load(f"{MODEL_DIR}/mlb_classes.npy", allow_pickle=True)
Y_test  = np.load(f"{MODEL_DIR}/test_labels.npy")
assert len(texts) == 725 and Y_test.shape == (725, 180) and len(classes) == 180

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok    = AutoTokenizer.from_pretrained(MODEL_DIR)
model  = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).float().to(device).eval()
assert model.config.num_labels == 180
assert tok.mask_token_id in tok(f"a {tok.mask_token} b")["input_ids"], "mask token is not tokenised as a single special token"

n_long = sum(len(tok(t)["input_ids"]) > MAX_LENGTH for t in texts)

METHOD = {"unit": "whitespace-delimited word",
          "masking": f"word replaced by {tok.mask_token}, surrounding whitespace kept",
          "algorithm": "permutation", "n_perm": N_PERM, "shap_seed": SHAP_SEED,
          "output": "logit of the top-1 predicted label", "max_length": MAX_LENGTH}

def logits_fn(batch_texts):
    """Raw logits (before the sigmoid), shape (len(batch_texts), 180)."""
    batch_texts = [str(t) for t in batch_texts]
    out = []
    with torch.no_grad():
        for s in range(0, len(batch_texts), 64):
            enc = tok(batch_texts[s:s + 64], truncation=True, max_length=MAX_LENGTH,
                      padding=True, return_tensors="pt").to(device)
            out.append(model(**enc).logits.cpu().numpy())
    return np.concatenate(out)

UNIT_RE = re.compile(r"(\s*)(\S+)(\s*)")

def split_units(text):
    """Split a sentence into (leading whitespace, word, trailing whitespace) triples that tile the text exactly."""
    parts = [m.groups() for m in UNIT_RE.finditer(text)]
    assert parts and "".join("".join(p) for p in parts) == text, "unit split does not reproduce the text"
    return parts

def build_text(parts, keep):
    """Sentence with every word whose keep flag is False replaced by the mask token."""
    return "".join(lead + (word if k else tok.mask_token) + trail for (lead, word, trail), k in zip(parts, keep))

def explain_row(row, shap_seed):
    text = texts[row]
    parts = split_units(text)
    n = len(parts)
    assert build_text(parts, [True] * n) == text, "unmasked reconstruction differs from the original text"
    full = logits_fn([text])[0]                          # all 180 logits for the original sentence
    k = int(full.argmax())                               # top-1 predicted label (ranking, no threshold)

    def f(M):                                            # M: (evaluations, n) matrix of 1 (word kept) / 0 (word masked)
        keep = np.asarray(M) > 0.5
        return logits_fn([build_text(parts, r) for r in keep])[:, k]

    explainer = shap.Explainer(f, shap.maskers.Independent(np.zeros((1, n))),
                               algorithm="permutation", seed=shap_seed)
    sv = explainer(np.ones((1, n)), max_evals=N_PERM * (2 * n + 1), batch_size=64, silent=True)[0]
    gold = [str(c) for c in classes[Y_test[row] == 1]]
    n_tok = len(tok(text)["input_ids"])
    return {"row": int(row), "text": text, "pred_label": str(classes[k]),
            "pred_prob": float(1 / (1 + np.exp(-full[k]))), "pred_logit": float(full[k]),
            "gold_labels": gold, "top1_correct": str(classes[k]) in gold,
            "units": ["".join(p) for p in parts], "shap_values": [float(v) for v in sv.values],
            "base_value": float(sv.base_values),
            "additivity_error": float(abs(sv.values.sum() + sv.base_values - full[k])),
            "n_model_tokens": n_tok, "truncated": n_tok > MAX_LENGTH, "method": METHOD}

def load_records(path):
    recs = {}
    if os.path.exists(path):
        for line in open(path):
            if line.strip():
                r = json.loads(line)
                assert r["method"] == METHOD, "stored records use different method settings; choose a new OUT_FILE"
                recs[r["row"]] = r
    return recs


In [ ]:
# Cell 3 - explain the selected sentences (SEED and N_SAMPLES are set in Cell 2)
order = [int(r) for r in np.random.RandomState(SEED).permutation(len(texts))[:N_SAMPLES]]
done = load_records(OUT_FILE)
todo = [r for r in order if r not in done]
print(f"shap {shap.__version__} | device={device} | selected rows: {len(order)} | stored: {len(order) - len(todo)} | to explain: {len(todo)}")
print(f"test sentences with more than {MAX_LENGTH} model tokens (the model reads only the first {MAX_LENGTH}): {n_long}")

t_start = time.time()
for i, row in enumerate(todo):
    t0 = time.time()
    rec = explain_row(row, SHAP_SEED)
    with open(OUT_FILE, "a") as fh:
        fh.write(json.dumps(rec) + "\n")
    print(f"[{i + 1}/{len(todo)}] row {row} | pred={rec['pred_label']} (p={rec['pred_prob']:.3f}) | gold={rec['gold_labels']} | "
          f"correct={rec['top1_correct']} | words={len(rec['units'])} | model_tokens={rec['n_model_tokens']} | "
          f"additivity_err={rec['additivity_error']:.1e} | {time.time() - t0:.1f}s")
    if i < PLOT_FIRST:
        try:
            shap.plots.text(shap.Explanation(values=np.array(rec["shap_values"]), base_values=rec["base_value"],
                                             data=np.array(rec["units"])))
        except Exception as e:
            print(f"  plot skipped ({type(e).__name__})")

recs = [r for r in load_records(OUT_FILE).values() if r["row"] in set(order)]
print(f"\n{len(recs)} selected sentences stored in {OUT_FILE}; top-1 correct: {sum(r['top1_correct'] for r in recs)} of {len(recs)}; "
      f"max additivity error: {max(r['additivity_error'] for r in recs):.1e}; time this run: {time.time() - t_start:.0f}s")
if any(r["additivity_error"] > 1e-3 for r in recs):
    print("WARNING: additivity error above 1e-3 for rows", [r["row"] for r in recs if r["additivity_error"] > 1e-3])


In [ ]:
# Cell 4 - sampling-noise check: re-explain the first CHECK_ROWS selected rows with another permutation seed
from scipy.stats import spearmanr
CHECK_ROWS = 20
stored = load_records(OUT_FILE)
rows_ok, rho, top3, top1 = [], [], [], []
for row in order[:CHECK_ROWS]:
    a, b = stored[row], explain_row(row, SHAP_SEED + 1)
    va, vb = np.array(a["shap_values"]), np.array(b["shap_values"])
    m = min(3, len(va))
    ta, tb = set(np.argsort(-np.abs(va))[:m]), set(np.argsort(-np.abs(vb))[:m])
    r = spearmanr(va, vb)[0] if len(va) > 2 and va.std() > 0 and vb.std() > 0 else np.nan
    rows_ok.append(row); rho.append(r); top3.append(len(ta & tb) / m); top1.append(int(np.argmax(np.abs(va)) == np.argmax(np.abs(vb))))
    print(f"row {row} | words={len(va)} | spearman={r:.3f} | top-3 overlap={top3[-1]:.2f} | same top-1 word={bool(top1[-1])}")
print(f"\nmean spearman={np.nanmean(rho):.3f} | mean top-3 overlap={np.mean(top3):.3f} | same top-1 word in {sum(top1)} of {len(top1)} rows")
